<a href="https://colab.research.google.com/github/scottaudrey/svd-food-recommendation-system/blob/main/notebooks/03_low_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part A: Load your project functions and data**

In [1]:
import os
import subprocess
import sys

USER = "scottaudrey"
REPO = "svd-food-recommendation-system"

if os.path.exists(REPO):
  subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
  subprocess.run(
    ["git", "clone", f"https://github.com/{USER}/{REPO}.git", REPO],
    check=True)

sys.path.insert(0, f"{REPO}/src")
from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline

This code connects to my GitHub repository and imports the reusable functions from Milestone 2. These functions will be used to split the ratings, create the rating matrix, calculate RMSE, and build the baseline model.

In [24]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://github.com/ryannewhouse11/FoodReviewData/releases/latest/download/recipereview.zip"
urllib.request.urlretrieve(url, "/content/recipereview.zip")

if not os.path.exists("/content/recipereview.zip"):
  urllib.request.urlretrieve(url, "/content/recipereview.zip")


with zipfile.ZipFile("/content/recipereview.zip", "r") as z:
  csv_files = [
      name for name in z.namelist()
      if name.lower().endswith(".csv") and not name.startswith("_MACOSX/")
  ]

  with z.open(csv_files[0]) as f:
    ratings = pd.read_csv(
        f,
        sep=",",
        header=0,
        names=["num_records", "recipe_number", "recipe_code", "recipe_name",
               "comment_id", "user_id", "user_name", "user_reputation", "created_at",
               "reply_count", "thumbs_up", "thumbs_down", "stars", "best_score", "text"]
    )

# This part is added to factor data and make these ID into consecutive integers
# This will help with Part B
# Code from chatgpt
ratings["user_id"] = pd.factorize(ratings["user_id"])[0] + 1
ratings["recipe_code"] = pd.factorize(ratings["recipe_code"])[0] + 1

train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)
print("fit:", len(fit), "validation:", len(val), "test:", len(test))

fit: 16983 validation: 560 test: 639


This code downloads the food review dataset and loads the ratings into a DataFrame. It removes missing values and converts user IDs and recipe codes into consecutive numbers. The data is then divided into fitting, validation, and testing sets. The fitting set is used to build the model, the validation set is used to compare models, and the testing set is saved for the final evaluation.

# **Part B: Set up a fair validation comparison**

In [25]:
R_fit = to_matrix(fit)
known_fit = ~np.isnan(R_fit)

val_truth = val["stars"].to_numpy()
val_u = val["user_id"].to_numpy() - 1
val_i = val["recipe_code"].to_numpy() - 1

def score_at(P, rows, cols, truth):
  """RMSE for selected entries of a prediction matrix."""
  pred = np.clip(P[rows, cols], 1, 5)
  return rmse(pred, truth)

This code creates a user-by-recipe rating matrix using only the fitting data. Missing ratings are represented by NaN. The validation ratings are kept separate to test how accurately the model predicts unknown ratings. The score_at function calculates RMSE by comparing predicted ratings to actual ratings. A lower RMSE means the predictions are more accurate.